<a href="https://colab.research.google.com/github/abdelrahmansaro/Ai_proj/blob/main/NYC_Collision_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYC Motor Vehicle Collision Analysis — Big Data Project
### Big Data, Spring 2026 — German International University
**Dr. Nada Sharaf**

---

**Team members:** *(fill in before submission)*
| Name | ID |
|------|----|
| _______________ | _______ |
| _______________ | _______ |
| _______________ | _______ |
| _______________ | _______ |

**Tools:** PySpark + SparkML on Google Colab
**Data window:** crash records from **2022-01-01 to 2024-12-31**
**Source:** [NYC Open Data](https://opendata.cityofnewyork.us/) — Motor Vehicle Collisions (Crashes / Vehicles / Persons), connected via `collision_id`.

---

### Notebook roadmap
0. Environment setup & Spark session
1. Data loading (read directly from the dataset links, filtered to 2022–2024)
2. Inspection (schema, counts, samples, missing-value summary, statistics)
3. Data cleaning & validation (every required issue handled and justified)
4. Data integration (crash-level, no fan-out duplication)
5. Feature engineering (time / location / severity / context / factor / **leakage-safe** historical + target)
6. Analytical queries (a–j) with business interpretation
7. SparkML — 3 classifiers predicting `future_hotspot`
8. Bonus — interactive dashboard

> **Reproducibility note:** all seeds fixed; every cell prints validation output so results are auditable.

## 0 · Environment Setup

Colab ships with Java, so we only install PySpark and the plotting libraries used in the dashboard.

In [ ]:
# Install PySpark + dashboard libraries (Colab already has Java 11/17)
!pip -q install pyspark==3.5.1 folium plotly 2>/dev/null
import os, sys
print("Python:", sys.version.split()[0])

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import IntegerType, DoubleType, StringType

# Spark session tuned for a single Colab VM. Bump driver memory if you have Colab Pro.
spark = (
    SparkSession.builder
    .appName("NYC_Collision_Analysis")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "ready")

## 1 · Data Loading

The datasets are large, so instead of downloading every year of data we **read directly from the dataset
links and filter to 2022–2024 at the source** using the Socrata SODA API (`$where` on `crash_date`).
This downloads only the rows we need.

**On the required filtering order.** The project mandates: filter Crashes by date → take its `collision_id`s
→ filter Vehicles and Persons by those IDs. We honour this exactly. We *additionally* pre-filter Vehicles
and Persons by `crash_date` at the API **only to reduce download size** — both datasets carry a `crash_date`
column. The *authoritative* filter is still the `collision_id` semi-join against the cleaned Crashes table,
applied in Spark in Section 4. (An API `IN(...)` clause over ~300k IDs is not feasible, hence the date
pre-filter as a download optimisation, not as the integration logic.)

> **Optional speed-up:** a free Socrata *app token* raises the throttling limit. Paste it below if you have
> one; otherwise leave it blank — the download still works, just a little slower.

In [ ]:
import requests

APP_TOKEN = ""   # optional: paste a Socrata app token to download faster (leave "" if none)

RESOURCES = {
    "crashes":  "h9gi-nx95",
    "vehicles": "bm4k-52h4",
    "persons":  "f55k-p6yu",
}
DATE_WHERE = "crash_date between '2022-01-01T00:00:00' and '2024-12-31T23:59:59'"

def socrata_download(resource_id, out_path, where=None, page=50000, app_token=""):
    """Paginated CSV download from the Socrata SODA API. Stable ordering via the internal :id."""
    base = f"https://data.cityofnewyork.us/resource/{resource_id}.csv"
    headers = {"X-App-Token": app_token} if app_token else {}
    offset, first, total = 0, True, 0
    with open(out_path, "w", encoding="utf-8") as f:
        while True:
            params = {"$limit": page, "$offset": offset, "$order": ":id"}
            if where:
                params["$where"] = where
            r = requests.get(base, params=params, headers=headers, timeout=300)
            r.raise_for_status()
            lines = r.text.splitlines()
            if len(lines) <= 1:                      # header only -> done
                break
            if first:
                f.write("\n".join(lines) + "\n"); first = False
            else:
                f.write("\n".join(lines[1:]) + "\n") # drop repeated header
            got = len(lines) - 1
            total += got; offset += got
            print(f"  {resource_id}: {total:,} rows", end="\r")
            if got < page:
                break
    print(f"  {resource_id}: {total:,} rows  -> {out_path}")
    return out_path

for name, rid in RESOURCES.items():
    print(f"Downloading {name} ...")
    socrata_download(rid, f"/content/{name}.csv", where=DATE_WHERE, app_token=APP_TOKEN)

In [ ]:
# Load into Spark DataFrames (read as strings first; we cast explicitly during cleaning)
crashes_raw  = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv("/content/crashes.csv")
vehicles_raw = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv("/content/vehicles.csv")
persons_raw  = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv("/content/persons.csv")

for nm, d in [("Crashes", crashes_raw), ("Vehicles", vehicles_raw), ("Persons", persons_raw)]:
    print(f"{nm}: {d.count():,} rows  x  {len(d.columns)} cols")

## 2 · Inspection

For each dataset we show: **schema, row count, column count, sample records, distinct `collision_id` count,
a missing-value summary, and basic statistics** — exactly as required before any cleaning decision.

In [ ]:
def inspect(df, name):
    print("="*70); print(f"  {name}"); print("="*70)
    print(f"Rows: {df.count():,}   Columns: {len(df.columns)}")
    print(f"Distinct collision_id: {df.select('collision_id').distinct().count():,}")
    print("\n-- Schema --")
    df.printSchema()
    print("-- Sample --")
    df.show(5, truncate=40)

inspect(crashes_raw, "CRASHES")

In [ ]:
inspect(vehicles_raw, "VEHICLES")

In [ ]:
inspect(persons_raw, "PERSONS")

In [ ]:
def missing_summary(df, name, cols=None):
    cols = cols or df.columns
    exprs = [F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in cols]
    row = df.agg(*exprs).collect()[0].asDict()
    n = df.count()
    print(f"\nMissing / blank values — {name}  (n={n:,})")
    for c, v in sorted(row.items(), key=lambda x: -x[1]):
        if v > 0:
            print(f"  {c:<40} {v:>10,}  ({100*v/n:5.1f}%)")

missing_summary(crashes_raw, "CRASHES")

In [ ]:
missing_summary(vehicles_raw, "VEHICLES")

In [ ]:
missing_summary(persons_raw, "PERSONS")

In [ ]:
# Basic statistics on numeric fields
num_cols = ["number_of_persons_injured","number_of_persons_killed",
            "number_of_pedestrians_injured","number_of_cyclist_injured",
            "number_of_motorist_injured","latitude","longitude"]
crashes_raw.select(num_cols).describe().show()
persons_raw.select("person_age").describe().show()

## 3 · Data Cleaning & Validation

We do **not** blindly delete missing values. Each required issue is handled with an explicit decision —
*removed, corrected, imputed, kept, or flagged* — and we validate after each major step.

| # | Issue | Decision | Why |
|---|-------|----------|-----|
| 1 | Missing/invalid crash date | **Removed** | Date is the analysis backbone (time features + monthly target); a crash with no date is unusable. |
| 2 | Missing crash time | **Imputed → 00:00 + flag** | Keeps the record for spatial/severity analysis; `time_missing_flag` lets us exclude it from hour-of-day work. |
| 3 | Missing borough / ZIP / street | **Kept + flagged** | Lat/long usually still present, so the crash is spatially usable; we flag rather than drop. |
| 4 | Missing latitude/longitude | **Kept, grid = NULL** | Borough/time analysis still valid; row is simply excluded from grid-based hotspot work. |
| 5 | Invalid coordinates (0,0 or outside NYC bbox) | **Corrected → NULL + flag** | Treated as "no location" so they don't pollute the grid. |
| 6 | Duplicate `collision_id` in Crashes | **Removed (keep one)** | One row must mean one crash; duplicates would double-count severity. |
| 7 | Unknown/unspecified contributing factor | **Kept + flagged** | "Unspecified" is itself information (data-quality signal); captured via `unknown_factor_flag`. |
| 8 | Negative / inconsistent injury & fatality counts | **Corrected → 0** | Counts cannot be negative; a negative is a data-entry error, floored to 0. |
| 9 | Vehicle rows without a matching crash | **Removed** | Enforced by the `collision_id` inner-join filter; orphans have no crash context. |
| 10 | Missing/unknown vehicle type | **Imputed → "UNKNOWN"** | Preserves the vehicle for counts while marking the gap. |
| 11 | Person rows without a matching crash | **Removed** | Same referential-integrity rule as vehicles. |
| 12 | Missing person type / invalid age / missing injury | **Imputed/Corrected/Flagged** | Type→"UNKNOWN"; ages outside 0–120 → NULL; injury→"Unspecified". |

In [ ]:
GRID_DECIMALS = 3   # ~110 m cells. Use 2 (~1.1 km) for coarser, denser grids if the monthly target is too sparse.

# ---- CRASHES: cast types ----
crashes = (crashes_raw
    .withColumn("crash_date", F.to_date(F.substring("crash_date", 1, 10), "yyyy-MM-dd"))
    .withColumn("latitude",  F.col("latitude").cast(DoubleType()))
    .withColumn("longitude", F.col("longitude").cast(DoubleType())))

count_cols = ["number_of_persons_injured","number_of_persons_killed",
              "number_of_pedestrians_injured","number_of_pedestrians_killed",
              "number_of_cyclist_injured","number_of_cyclist_killed",
              "number_of_motorist_injured","number_of_motorist_killed"]
for c in count_cols:
    crashes = crashes.withColumn(c, F.col(c).cast(IntegerType()))

print("Before cleaning:", crashes.count())

# (1) Re-assert the 2022-2024 date window in Spark + drop missing/invalid dates  -> REMOVED
crashes = crashes.filter(F.col("crash_date").isNotNull())
crashes = crashes.filter((F.col("crash_date") >= F.lit("2022-01-01")) &
                         (F.col("crash_date") <= F.lit("2024-12-31")))
print("After date filter :", crashes.count())

# (6) Duplicate collision_id  -> REMOVED (keep one row per crash)
dups = crashes.count() - crashes.dropDuplicates(["collision_id"]).count()
crashes = crashes.dropDuplicates(["collision_id"])
print(f"Duplicate collision_ids removed: {dups:,}")

# (2) Missing crash time -> IMPUTED 00:00 + FLAG
crashes = crashes.withColumn("time_missing_flag",
            F.when(F.col("crash_time").isNull() | (F.trim("crash_time") == ""), 1).otherwise(0))
crashes = crashes.withColumn("crash_time", F.coalesce(F.col("crash_time"), F.lit("00:00")))

# (5) Invalid coordinates -> NULL + FLAG (NYC bounding box)
crashes = crashes.withColumn("coord_valid",
            (F.col("latitude").between(40.45, 40.95)) & (F.col("longitude").between(-74.30, -73.65)))
crashes = (crashes
    .withColumn("latitude",  F.when(F.col("coord_valid"), F.col("latitude")))
    .withColumn("longitude", F.when(F.col("coord_valid"), F.col("longitude"))))

# (3)(4) Missing borough/zip/street -> KEEP + FLAG
crashes = crashes.withColumn("borough_missing_flag",
            F.when(F.col("borough").isNull() | (F.trim("borough") == ""), 1).otherwise(0))

# (8) Negative / null injury & fatality counts -> CORRECTED to 0
for c in count_cols:
    crashes = crashes.withColumn(c, F.when(F.col(c) < 0, 0).otherwise(F.coalesce(F.col(c), F.lit(0))))

# (7) Unknown contributing factor -> KEEP + FLAG (built in Section 5)
crashes.cache()
print("Crashes after cleaning:", crashes.count(),
      "| distinct ids:", crashes.select('collision_id').distinct().count())

In [ ]:
# ---- Referential integrity: keep only Vehicles/Persons whose collision_id exists in cleaned Crashes ----
valid_ids = crashes.select("collision_id").distinct()

veh_before, per_before = vehicles_raw.count(), persons_raw.count()
vehicles = vehicles_raw.join(F.broadcast(valid_ids), "collision_id", "inner")   # (9) orphans REMOVED
persons  = persons_raw.join(F.broadcast(valid_ids), "collision_id", "inner")    # (11) orphans REMOVED

# (10) Missing/unknown vehicle type -> IMPUTED "UNKNOWN"
vehicles = vehicles.withColumn("vehicle_type",
    F.when(F.col("vehicle_type").isNull() | (F.trim("vehicle_type") == ""), "UNKNOWN")
     .otherwise(F.upper(F.trim("vehicle_type"))))

# (12) Person fields: type -> UNKNOWN, age 0-120 else NULL, injury -> Unspecified
persons = persons.withColumn("person_age", F.col("person_age").cast(IntegerType()))
persons = persons.withColumn("person_age",
            F.when((F.col("person_age") < 0) | (F.col("person_age") > 120), None).otherwise(F.col("person_age")))
persons = persons.withColumn("person_type",
            F.when(F.col("person_type").isNull() | (F.trim("person_type") == ""), "UNKNOWN").otherwise(F.col("person_type")))
persons = persons.withColumn("person_injury",
            F.when(F.col("person_injury").isNull() | (F.trim("person_injury") == ""), "Unspecified").otherwise(F.col("person_injury")))

print(f"Vehicles: {veh_before:,} -> {vehicles.count():,}  (orphans removed: {veh_before-vehicles.count():,})")
print(f"Persons : {per_before:,} -> {persons.count():,}  (orphans removed: {per_before-persons.count():,})")

## 4 · Data Integration

**Strategy — keep the data at crash level.** A naive three-way join (Crashes ⋈ Vehicles ⋈ Persons on
`collision_id`) would *fan out*: a crash with 3 vehicles and 5 people becomes 15 rows, multiplying every
crash-level value and corrupting counts and the ML target.

Instead we **aggregate Vehicles and Persons up to one row per `collision_id`** (vehicle counts/type flags,
person counts, age stats, vulnerable-user counts) and **left-join those summaries onto the Crashes base.**
The result is exactly one row per crash — analysis-ready and ML-ready — with the richer vehicle/person
context preserved as engineered columns.

In [ ]:
# Aggregate VEHICLES -> one row per collision_id
veh_agg = (vehicles.groupBy("collision_id").agg(
    F.countDistinct("unique_id").alias("num_vehicles"),
    F.collect_set("vehicle_type").alias("_vtypes"),
))
veh_agg = (veh_agg
    .withColumn("has_truck",      F.expr("exists(_vtypes, x -> x rlike 'TRUCK|TRACTOR|BOX')").cast("int"))
    .withColumn("has_motorcycle", F.expr("exists(_vtypes, x -> x rlike 'MOTORCYCLE|MOPED')").cast("int"))
    .withColumn("has_bicycle",    F.expr("exists(_vtypes, x -> x rlike 'BIKE|BICYCLE')").cast("int"))
    .withColumn("multi_vehicle",  F.when(F.col("num_vehicles") >= 2, 1).otherwise(0))
    .drop("_vtypes"))

# Aggregate PERSONS -> one row per collision_id
per_agg = (persons.groupBy("collision_id").agg(
    F.count("*").alias("num_persons"),
    F.sum(F.when(F.col("person_type") == "Pedestrian", 1).otherwise(0)).alias("num_pedestrians"),
    F.sum(F.when(F.col("person_type") == "Bicyclist", 1).otherwise(0)).alias("num_cyclists"),
    F.round(F.avg("person_age"), 1).alias("avg_person_age"),
    F.sum(F.when(F.col("person_age") < 18, 1).otherwise(0)).alias("num_minors"),
    F.sum(F.when(F.col("person_age") >= 65, 1).otherwise(0)).alias("num_elderly"),
))

# Integrate (LEFT join keeps every crash even if a vehicle/person row is absent)
df = (crashes
      .join(veh_agg, "collision_id", "left")
      .join(per_agg, "collision_id", "left")
      .fillna({"num_vehicles":0,"has_truck":0,"has_motorcycle":0,"has_bicycle":0,"multi_vehicle":0,
               "num_persons":0,"num_pedestrians":0,"num_cyclists":0,"num_minors":0,"num_elderly":0}))
df.cache()

print("Final row count            :", df.count())
print("Final distinct collision_id:", df.select('collision_id').distinct().count())
print("Duplicate collision_id     :", df.groupBy('collision_id').count().filter('count > 1').count())

## 5 · Feature Engineering

We build six families of features. **Severity** uses the weighted score from the brief. The **historical**
features and the **`future_hotspot`** target are built on a complete *grid × month* calendar panel so that
lags reference real calendar months (not just the previous *observed* row).

> **No leakage:** historical features use **past months only** (`lag` / trailing windows that exclude the
> current month). The target uses the **next** month only. The two never overlap.

In [ ]:
# ---------- 5.1 Time features ----------
df = df.withColumn("crash_datetime",
        F.to_timestamp(F.concat_ws(" ", F.col("crash_date").cast("string"),
                                    F.coalesce(F.col("crash_time"), F.lit("00:00"))), "yyyy-MM-dd H:mm"))
df = (df
    .withColumn("hour",        F.coalesce(F.hour("crash_datetime"), F.lit(0)))
    .withColumn("day_of_week", F.date_format("crash_date", "EEEE"))
    .withColumn("_dow",        F.dayofweek("crash_date"))            # 1=Sun ... 7=Sat
    .withColumn("is_weekend",  F.when(F.col("_dow").isin(1, 7), 1).otherwise(0))
    .withColumn("month",       F.month("crash_date"))
    .withColumn("year",        F.year("crash_date"))
    .withColumn("year_month",  F.date_format("crash_date", "yyyy-MM"))
    .withColumn("season",      F.when(F.col("month").isin(12,1,2), "Winter")
                                .when(F.col("month").isin(3,4,5),  "Spring")
                                .when(F.col("month").isin(6,7,8),  "Summer").otherwise("Fall"))
    .withColumn("period_of_day", F.when(F.col("hour").between(5,11),  "Morning")
                                  .when(F.col("hour").between(12,16), "Afternoon")
                                  .when(F.col("hour").between(17,20), "Evening").otherwise("Night"))
    .withColumn("rush_hour_flag", F.when((F.col("is_weekend") == 0) &
                  (F.col("hour").between(7,9) | F.col("hour").between(16,18)), 1).otherwise(0))
    .withColumn("night_flag",   F.when((F.col("hour") >= 20) | (F.col("hour") <= 5), 1).otherwise(0)))

In [ ]:
# ---------- 5.2 Location features ----------
df = df.withColumn("location_grid",
        F.when(F.col("coord_valid"),
               F.concat_ws("_", F.round("latitude", GRID_DECIMALS), F.round("longitude", GRID_DECIMALS))))

grid_counts = (df.filter(F.col("location_grid").isNotNull())
                 .groupBy("location_grid").count()
                 .withColumnRenamed("count", "crash_count_per_grid"))
df = df.join(grid_counts, "location_grid", "left")

grid_thr = grid_counts.approxQuantile("crash_count_per_grid", [0.75], 0.01)[0]
df = df.withColumn("street_hotspot_flag",
        F.when(F.col("crash_count_per_grid") >= F.lit(grid_thr), 1).otherwise(0))
print(f"Grid 75th-pct crash count threshold = {grid_thr}")

In [ ]:
# ---------- 5.3 Severity features ----------
# severity_score = 1*injured + 3*ped_injured + 3*cyc_injured + 5*killed   (formula from the brief)
df = (df
    .withColumn("total_injured", F.col("number_of_persons_injured"))
    .withColumn("total_killed",  F.col("number_of_persons_killed"))
    .withColumn("severity_score",
            F.col("number_of_persons_injured")
          + 3*F.col("number_of_pedestrians_injured")
          + 3*F.col("number_of_cyclist_injured")
          + 5*F.col("number_of_persons_killed"))
    .withColumn("severity_level",
            F.when(F.col("total_killed") > 0, "Fatal")
             .when(F.col("severity_score") >= 6, "Severe")
             .when(F.col("severity_score") >= 1, "Moderate").otherwise("Minor")))

In [ ]:
# ---------- 5.4 Collision-context features (already merged in Section 4) ----------
# num_vehicles, multi_vehicle, has_truck/motorcycle/bicycle, num_persons,
# num_pedestrians, num_cyclists, avg_person_age, num_minors, num_elderly.
df = df.withColumn("vulnerable_user_involved",
        F.when((F.col("num_pedestrians") > 0) | (F.col("num_cyclists") > 0), 1).otherwise(0))

In [ ]:
# ---------- 5.5 Contributing-factor features ----------
f1 = F.col("contributing_factor_vehicle_1")
df = (df
    .withColumn("driver_behavior_flag", F.when(f1.rlike(
        "Inattention|Distraction|Speed|Following|Disregarded|Aggressive|Improper|Passing|Fatigued|Drowsy|Alcohol|Drug|Yield|Backing|Turning"), 1).otherwise(0))
    .withColumn("environment_related_flag", F.when(f1.rlike(
        "Slippery|Glare|Obstruct|View|Lighting|Weather|Pavement|Snow|Ice"), 1).otherwise(0))
    .withColumn("vehicle_defect_flag", F.when(f1.rlike(
        "Brakes|Tire|Defective|Steering|Headlights|Accelerator|Defect|Failure"), 1).otherwise(0))
    .withColumn("unknown_factor_flag", F.when(
        f1.isNull() | (F.trim(f1) == "") | (f1 == "Unspecified"), 1).otherwise(0)))
df = df.withColumn("primary_factor_group",
        F.when(F.col("vehicle_defect_flag") == 1, "Vehicle Defect")
         .when(F.col("environment_related_flag") == 1, "Environment")
         .when(F.col("driver_behavior_flag") == 1, "Driver Behavior")
         .when(F.col("unknown_factor_flag") == 1, "Unknown").otherwise("Other"))
df.cache()
print("Crash-level engineered dataset:", df.count(), "rows,", len(df.columns), "columns")

### 5.6 · Grid × Month panel — historical features + `future_hotspot` target

We aggregate crashes to `location_grid` + month, then build a **complete calendar panel** (every grid ×
every month 2022-01…2024-12, missing months filled with 0). On this panel:

* **Historical (past-only):** `crashes_previous_month` = last month's count; `crashes_previous_3_months` =
  sum of the 3 months *before* the current one; `avg_severity_previous_month / _3_months` likewise.
* **Target:** a grid is a *hotspot this month* if its monthly crash count **or** severity sum is in the
  **top 25%** of active grids that month. `future_hotspot` = whether the grid is a hotspot **next** month.

In [ ]:
gm = df.filter(F.col("location_grid").isNotNull()).withColumn(
        "m_idx", (F.col("year") - 2022) * 12 + (F.col("month") - 1))   # 0..35

monthly = (gm.groupBy("location_grid", "m_idx").agg(
    F.count("*").alias("crash_count"),
    F.sum("severity_score").alias("sev_sum"),
    F.round(F.avg("severity_score"), 2).alias("sev_avg"),
    F.sum("total_killed").alias("killed_sum"),
    F.sum("num_pedestrians").alias("ped_sum"),
    F.sum("num_cyclists").alias("cyc_sum")))

# complete panel: every active grid x every month
grids  = monthly.select("location_grid").distinct()
months = spark.range(0, 36).withColumnRenamed("id", "m_idx")
panel  = grids.crossJoin(months).join(monthly, ["location_grid", "m_idx"], "left").fillna(0)

wg = W.partitionBy("location_grid").orderBy("m_idx")
panel = (panel
    .withColumn("crashes_previous_month",        F.lag("crash_count", 1).over(wg))
    .withColumn("crashes_previous_3_months",     F.sum("crash_count").over(wg.rowsBetween(-3, -1)))
    .withColumn("avg_severity_previous_month",   F.lag("sev_avg", 1).over(wg))
    .withColumn("avg_severity_previous_3_months",F.avg("sev_avg").over(wg.rowsBetween(-3, -1))))
panel = panel.fillna(0, subset=["crashes_previous_month","crashes_previous_3_months",
                                "avg_severity_previous_month","avg_severity_previous_3_months"])

# monthly top-25% thresholds among ACTIVE grids -> hotspot-this-month label
thr = (panel.filter(F.col("crash_count") > 0).groupBy("m_idx").agg(
        F.expr("percentile_approx(crash_count, 0.75)").alias("cnt_thr"),
        F.expr("percentile_approx(sev_sum, 0.75)").alias("sev_thr")))
panel = panel.join(thr, "m_idx", "left").withColumn("is_hotspot_this_month",
        F.when((F.col("crash_count") > 0) &
               ((F.col("crash_count") >= F.col("cnt_thr")) | (F.col("sev_sum") >= F.col("sev_thr"))), 1).otherwise(0))

# TARGET = next month's hotspot status
panel = panel.withColumn("future_hotspot", F.lead("is_hotspot_this_month", 1).over(wg))
panel.cache()

# Sanity check on the busiest grid (previous_month must equal last month's crash_count)
busy = panel.groupBy("location_grid").agg(F.sum("crash_count").alias("t")).orderBy(F.desc("t")).first()["location_grid"]
print("Leakage/lag check on busiest grid", busy, ":")
panel.filter(F.col("location_grid") == busy).select(
    "m_idx","crash_count","crashes_previous_month","crashes_previous_3_months",
    "is_hotspot_this_month","future_hotspot").orderBy("m_idx").show(8)

In [ ]:
# Attach the (grid, month) future_hotspot back onto every crash row
df = df.withColumn("m_idx", (F.col("year") - 2022) * 12 + (F.col("month") - 1))
df = df.join(panel.select("location_grid", "m_idx", "future_hotspot"),
             ["location_grid", "m_idx"], "left")
df.cache()
print("future_hotspot attached to crash-level data.")
df.groupBy("future_hotspot").count().orderBy("future_hotspot").show()

## 6 · Analytical Queries (a–j)

Each query uses the Spark DataFrame API on the integrated crash-level table, followed by a short
**business interpretation** for transportation-safety / city-planning decisions.

### (a) Boroughs with the most crashes, and the trend over time

In [ ]:
(df.filter(F.col("borough").isNotNull() & (F.trim("borough") != ""))
   .groupBy("borough", "year").count()
   .groupBy("borough").pivot("year").sum("count")
   .orderBy(F.desc("2024"))).show()

**Interpretation:** ranks boroughs by crash volume and shows whether each is rising or falling year-on-year.
A borough that is both high-volume **and** trending up is the priority for enforcement and infrastructure
budget; a falling trend can validate interventions already deployed there.

### (b) Highest injury & fatality **rates** (not just counts)

In [ ]:
(df.filter(F.col("on_street_name").isNotNull() & (F.trim("on_street_name") != ""))
   .groupBy("on_street_name").agg(
       F.count("*").alias("crashes"),
       F.sum("total_injured").alias("injured"),
       F.sum("total_killed").alias("killed"),
       F.round(F.sum("total_injured")/F.count("*"), 3).alias("injury_rate"),
       F.round(F.sum("total_killed")/F.count("*"), 4).alias("fatality_rate"))
   .filter(F.col("crashes") >= 50)            # min volume so rates are meaningful
   .orderBy(F.desc("fatality_rate"), F.desc("injury_rate"))).show(15, truncate=False)

**Interpretation:** a street can have few crashes but a high share that injure or kill — these are
*dangerous-by-design* corridors (speed, road geometry) rather than merely *busy* ones. Rate-based ranking
targets redesign (traffic calming, signals) where each crash is most harmful, not just where crashes are frequent.

### (c) Hours and days of week with the most crashes

In [ ]:
(df.groupBy("day_of_week", "hour").count()
   .orderBy(F.desc("count"))).show(12)
print("By day of week:")
(df.groupBy("day_of_week").count().orderBy(F.desc("count"))).show()

**Interpretation:** pinpoints the day/hour windows for the largest crash burden — directly actionable for
scheduling police enforcement, deploying traffic agents, and timing public-awareness campaigns.

### (d) Rush-hour vs non-rush-hour crash counts and severity

In [ ]:
(df.groupBy("rush_hour_flag").agg(
    F.count("*").alias("crashes"),
    F.round(F.avg("severity_score"), 3).alias("avg_severity"),
    F.sum("total_killed").alias("killed"))
   .orderBy("rush_hour_flag")).show()

**Interpretation:** distinguishes *congestion* crashes (rush hour — many crashes, often low speed/low
severity) from *off-peak* crashes (fewer but potentially more severe due to higher speeds). The severity
column tells planners whether to prioritise throughput fixes or speed management.

### (e) Weekday vs weekend: frequency, severity, vulnerable-user involvement

In [ ]:
(df.groupBy("is_weekend").agg(
    F.count("*").alias("crashes"),
    F.round(F.avg("severity_score"), 3).alias("avg_severity"),
    F.round(F.avg("vulnerable_user_involved"), 3).alias("share_vulnerable"),
    F.sum("num_pedestrians").alias("peds"),
    F.sum("num_cyclists").alias("cyclists"))
   .orderBy("is_weekend")).show()

**Interpretation:** weekends often show fewer crashes but a higher *share* involving pedestrians/cyclists
and higher severity (nightlife, alcohol, recreational walking/cycling). Guides weekend-specific measures
such as targeted DUI checkpoints and protected pedestrian zones.

### (f) Seasonal variation in crash count and severity

In [ ]:
(df.groupBy("season").agg(
    F.count("*").alias("crashes"),
    F.round(F.avg("severity_score"), 3).alias("avg_severity"))
   .orderBy(F.desc("crashes"))).show()

**Interpretation:** seasonality (e.g. winter road conditions, summer traffic volume) supports seasonal
resource planning — salting/plowing readiness, seasonal signage, and timing of safety campaigns.

### (g) Contributing-factor groups most associated with severe crashes

In [ ]:
(df.groupBy("primary_factor_group").agg(
    F.count("*").alias("crashes"),
    F.round(F.avg("severity_score"), 3).alias("avg_severity"),
    F.sum(F.when(F.col("severity_level").isin("Severe","Fatal"), 1).otherwise(0)).alias("severe_or_fatal"))
   .orderBy(F.desc("avg_severity"))).show(truncate=False)

**Interpretation:** links cause to outcome. If "Driver Behavior" dominates severe crashes, the lever is
enforcement/education; if "Environment" does, it's road maintenance and lighting. Directs the *type* of
intervention, not just the location.

### (h) Vehicle types most frequently involved in injury crashes

In [ ]:
inj = df.filter(F.col("total_injured") > 0).select("collision_id")
(vehicles.join(inj, "collision_id", "inner")
         .groupBy("vehicle_type").count()
         .orderBy(F.desc("count"))).show(15, truncate=False)

**Interpretation:** identifies vehicle categories over-represented in injury crashes (e.g. SUVs, trucks,
e-bikes). Informs fleet regulation, commercial-vehicle routing rules, and micro-mobility policy.

### (i) Locations with the most pedestrian & cyclist injuries

In [ ]:
(df.filter(F.col("location_grid").isNotNull()).groupBy("location_grid").agg(
    F.sum("number_of_pedestrians_injured").alias("ped_injured"),
    F.sum("number_of_cyclist_injured").alias("cyc_injured"),
    (F.sum("number_of_pedestrians_injured") + F.sum("number_of_cyclist_injured")).alias("vru_injured"))
   .orderBy(F.desc("vru_injured"))).show(15, truncate=False)

**Interpretation:** the precise grid cells where vulnerable road users are hurt most — the strongest case
for protected bike lanes, pedestrian islands, leading pedestrian signals, and lower speed limits.

### (j) High-risk hotspots by crash frequency **and** severity

In [ ]:
(df.filter(F.col("location_grid").isNotNull()).groupBy("location_grid").agg(
    F.count("*").alias("crashes"),
    F.sum("severity_score").alias("total_severity"),
    F.round(F.avg("severity_score"), 2).alias("avg_severity"))
   .withColumn("risk_score", F.col("crashes") * F.col("avg_severity"))
   .orderBy(F.desc("risk_score"))).show(15, truncate=False)

**Interpretation:** combines *how often* and *how badly* into one risk score, so a grid is flagged whether
it is high-frequency/low-severity or low-frequency/high-severity. This is the ranked worklist for Vision-Zero
style site investigations.

## 7 · SparkML — Predicting `future_hotspot`

**Unit of analysis:** one row per `location_grid` + `month`. **Target:** `future_hotspot` (will this grid be
a top-25% hotspot *next* month?).

**Modelling choices**
* We train only on **active grid-months** (a crash this month *or* in the trailing 3 months). Grids that have
  never seen activity make the task degenerate and swamp the negatives, so they are excluded.
* The target is **imbalanced**, so Logistic Regression uses **class weights**; we report **precision, recall,
  F1 and the confusion matrix** (not just accuracy) and **AUC**, because accuracy alone is misleading here.
* Pipeline: `StringIndexer`/`OneHotEncoder` (for any categorical), `VectorAssembler`, `StandardScaler`,
  then the classifier — all inside a Spark `Pipeline` so the same transforms apply to train and test.

In [ ]:
# ---- Build the monthly grid-level modelling table ----
ml = (panel
      .filter(F.col("m_idx") < 35)                        # last month has no "next month" -> no label
      .filter(F.col("future_hotspot").isNotNull())
      .filter((F.col("crash_count") > 0) | (F.col("crashes_previous_3_months") > 0)))  # active grids only

ml = ml.withColumn("label", F.col("future_hotspot").cast("double"))
ml = ml.withColumn("season_idx",                         # a categorical feature to demo encoding
        F.when((F.col("m_idx") % 12).isin(11,0,1), "Winter")
         .when((F.col("m_idx") % 12).isin(2,3,4),  "Spring")
         .when((F.col("m_idx") % 12).isin(5,6,7),  "Summer").otherwise("Fall"))

print("Modelling rows:", ml.count())
ml.groupBy("label").count().orderBy("label").show()

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

num_features = ["crash_count","crashes_previous_month","crashes_previous_3_months",
                "avg_severity_previous_month","avg_severity_previous_3_months",
                "sev_sum","ped_sum","cyc_sum","m_idx"]

# Categorical encoding (StringIndexer + OneHotEncoder) as required
idx = StringIndexer(inputCol="season_idx", outputCol="season_ix", handleInvalid="keep")
ohe = OneHotEncoder(inputCols=["season_ix"], outputCols=["season_oh"])

assembler = VectorAssembler(inputCols=num_features + ["season_oh"], outputCol="features_raw")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# class weight for imbalance
n_neg = ml.filter(F.col("label") == 0).count()
n_pos = ml.filter(F.col("label") == 1).count()
weight = n_neg / max(n_pos, 1)
ml = ml.withColumn("w", F.when(F.col("label") == 1.0, float(weight)).otherwise(1.0))
print(f"Class balance  neg={n_neg:,}  pos={n_pos:,}  -> positive weight={weight:.2f}")

train, test = ml.randomSplit([0.8, 0.2], seed=42)
print(f"Train={train.count():,}  Test={test.count():,}")

In [ ]:
def run_model(name, classifier):
    pipe = Pipeline(stages=[idx, ohe, assembler, scaler, classifier])
    model = pipe.fit(train)
    pred  = model.transform(test)
    acc  = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred)
    f1   = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred)
    prec = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(pred)
    rec  = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(pred)
    auc  = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(pred)
    print(f"\n=== {name} ===")
    print(f"Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  AUC={auc:.3f}")
    print("Confusion matrix (rows=actual, cols=predicted):")
    (pred.groupBy("label","prediction").count().orderBy("label","prediction")).show()
    return {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc, "fitted": model}

lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="w", maxIter=100)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=6)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=8, seed=42)

results = [run_model("Logistic Regression", lr),
           run_model("Decision Tree",       dt),
           run_model("Random Forest",       rf)]

In [ ]:
import pandas as pd
summary = pd.DataFrame([{k: r[k] for k in ["model","accuracy","precision","recall","f1","auc"]} for r in results])
summary = summary.sort_values("f1", ascending=False).reset_index(drop=True)
print("MODEL COMPARISON (sorted by F1)")
display(summary.round(3))
best = summary.iloc[0]["model"]
print(f"\nBest model by F1: {best}")

**Discussion.** We compare the three models on F1 and AUC rather than raw accuracy, because with an
imbalanced target a "predict-not-hotspot" baseline already scores high accuracy while being useless.

* **Logistic Regression** (class-weighted) is the interpretable linear baseline — strong recall on the
  positive class thanks to the weighting, but it can over-predict hotspots (lower precision).
* **Decision Tree** captures non-linear thresholds (e.g. "≥ N crashes last month") and is easy to explain to
  city officials, but a single tree can overfit.
* **Random Forest** typically gives the best, most stable F1/AUC because it averages many trees and handles
  feature interactions, at the cost of interpretability.

The recurring, intuitively-correct driver is recent history: `crashes_previous_month` and
`crashes_previous_3_months` — a grid that was busy/severe recently tends to stay a hotspot. *(Exact numbers
depend on the live data; read them off the comparison table above.)*

## 8 · Bonus — Interactive Dashboard

Four visualisations built from the Spark query outputs, plus interactivity:
1. **Borough crash trend** over months — with an **interactive borough dropdown filter** (Plotly).
2. **Hour × day-of-week heatmap** of crash volume.
3. **Severity by contributing-factor group** (bar).
4. **Hotspot map** — top risk grids on a Folium map.

We pull the (small) aggregated results into pandas for plotting; the heavy lifting stays in Spark.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# --- aggregates -> pandas ---
trend_pdf = (df.filter(F.col("borough").isNotNull() & (F.trim("borough") != ""))
               .groupBy("year_month","borough").count().orderBy("year_month")).toPandas()

# VISUAL 1: interactive borough filter via dropdown
boroughs = sorted(trend_pdf["borough"].unique())
fig1 = go.Figure()
for b in boroughs:
    sub = trend_pdf[trend_pdf.borough == b].sort_values("year_month")
    fig1.add_trace(go.Scatter(x=sub.year_month, y=sub["count"], name=b, mode="lines+markers"))
buttons = [dict(label="All", method="update", args=[{"visible":[True]*len(boroughs)}])]
for i,b in enumerate(boroughs):
    vis=[j==i for j in range(len(boroughs))]
    buttons.append(dict(label=b, method="update", args=[{"visible":vis}]))
fig1.update_layout(title="Monthly crashes by borough (interactive filter)",
    updatemenus=[dict(buttons=buttons, x=1.15, y=1)], xaxis_title="Month", yaxis_title="Crashes", height=450)
fig1.show()

In [ ]:
# VISUAL 2: hour x day-of-week heatmap
hd = (df.groupBy("day_of_week","hour").count()).toPandas()
order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
pivot = hd.pivot_table(index="day_of_week", columns="hour", values="count", fill_value=0).reindex(order)
fig2 = px.imshow(pivot, aspect="auto", color_continuous_scale="Reds",
                 labels=dict(x="Hour of day", y="Day", color="Crashes"),
                 title="Crash volume by hour and day of week")
fig2.update_layout(height=420); fig2.show()

In [ ]:
# VISUAL 3: severity by contributing-factor group
fg = (df.groupBy("primary_factor_group").agg(
        F.count("*").alias("crashes"),
        F.round(F.avg("severity_score"),3).alias("avg_severity"))).toPandas()
fig3 = px.bar(fg.sort_values("avg_severity"), x="primary_factor_group", y="avg_severity",
              color="crashes", title="Average severity by contributing-factor group",
              labels={"primary_factor_group":"Factor group","avg_severity":"Avg severity score"})
fig3.update_layout(height=420); fig3.show()

In [ ]:
# VISUAL 4: hotspot map (top risk grids)
import folium
hot = (df.filter(F.col("location_grid").isNotNull()).groupBy("location_grid").agg(
        F.count("*").alias("crashes"),
        F.sum("severity_score").alias("sev"),
        F.avg("latitude").alias("lat"), F.avg("longitude").alias("lon"))
       .withColumn("risk", F.col("crashes")*F.col("sev"))
       .orderBy(F.desc("risk")).limit(150)).toPandas()

m = folium.Map(location=[40.71,-73.94], zoom_start=11, tiles="cartodbpositron")
mx = hot["risk"].max() or 1
for _,r in hot.iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=4+10*(r.risk/mx),
        color="crimson", fill=True, fill_opacity=0.6,
        popup=f"{r.location_grid}<br>crashes={int(r.crashes)} severity={int(r.sev)}").add_to(m)
m

---
### Submission checklist
- [ ] Fill in team member names & IDs (title cell).
- [ ] **Run all cells top-to-bottom** so every output, table, query result, model metric and chart is saved.
- [ ] Email the notebook (+ any dashboard screenshots) as a zip to **sandra.samuel@giu-berlin.de**.
- [ ] Subject: **"Project – NYC Traffic Collision Analysis"**; include all team names & IDs in the body.
- [ ] Be ready to justify every cleaning, feature, and model decision (see the markdown rationale above).